
# Hippo cell-level CSV export notebook

Run this notebook in the Python environment where your `adata` is already loaded.

It exports CSVs that can be sent back for plotting:
- cell-state composition by target and guide
- target-level state enrichment
- guide-level fate-axis effects
- cell-level score distributions for Hippo targets
- UMAP overlay coordinates for Hippo carriers
- guide convergence summary


In [ ]:

import os
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import fisher_exact, ks_2samp, mannwhitneyu, wasserstein_distance



## 1. User settings


In [ ]:

OUTDIR = "hippo_nature_style_exports"
GUIDE_TARGET_CSV = "combined_TargetKD.csv"

CNMF_KEY = "cnmf_usage"
GUIDE_ASSIGNMENT_KEY = "guide_assignment"

DAY_COL = "day"
CELLTYPE_COL = "cell_type"

HIPPO_CORE = [
    "NF2", "LATS1", "LATS2", "SAV1", "STK3", "STK4",
    "MOB1A", "MOB1B", "TAOK1", "TAOK2", "TAOK3",
    "AMOT", "AMOTL1", "AMOTL2"
]

PROGRAMS = [2, 3, 8, 11, 13, 15, 17, 23, 27, 30, 34, 35]

D2_PRE_EC = "Pre-EC"
D2_ALT = "Lat-Meso"

D3_EC_LABELS = ["artEC", "ECM-EC"]
D3_ALT_LABELS = ["Fibro-Mesen", "Neu-Mesen"]
D3_MES_LABELS = ["Fibro-Mesen"]

os.makedirs(OUTDIR, exist_ok=True)



## 2. Helper functions


In [ ]:

def as_dense_bool(x):
    if sparse.issparse(x):
        x = x.toarray()
    return np.asarray(x) > 0

def clean_day(x):
    return str(x).strip().lower()

def looks_nt(x):
    s = str(x).lower()
    patterns = [
        "non-target", "nontarget", "non_target", "negative", "control",
        "scramble", "scrambled", "safe", "ntc", "non targeting", "non-targeting"
    ]
    return any(p in s for p in patterns) or s in {"nt", "neg", "ctrl"}

def infer_col(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    for c in df.columns:
        cl = c.lower()
        for cand in candidates:
            if cand.lower() in cl:
                return c
    return None

def parse_target_from_guide_name(guide):
    g = str(guide)
    for sep in ["|", ":", "_", "-"]:
        if sep in g:
            return g.split(sep)[0]
    return g

def build_guide_target_table(guide_names, guide_target_csv):
    guide_names = pd.Index([str(g) for g in guide_names], name="guide")
    base = pd.DataFrame({"guide": guide_names})

    if guide_target_csv is not None and os.path.exists(guide_target_csv):
        gt = pd.read_csv(guide_target_csv)
        guide_col = infer_col(gt, ["guide", "guide_name", "sgRNA", "sgrna", "gRNA", "oligo", "spacer"])
        target_col = infer_col(gt, ["target", "target_name", "target_gene", "gene", "symbol", "Target", "TargetGene"])
        if guide_col is None or target_col is None:
            raise ValueError(f"Could not infer guide/target columns. Columns found: {list(gt.columns)}")
        gt = gt[[guide_col, target_col]].copy()
        gt.columns = ["guide", "target"]
        gt["guide"] = gt["guide"].astype(str)
        gt["target"] = gt["target"].astype(str)
        out = base.merge(gt, on="guide", how="left")
        out["target"] = out["target"].fillna(out["guide"].map(parse_target_from_guide_name))
    else:
        out = base.copy()
        out["target"] = out["guide"].map(parse_target_from_guide_name)

    out["is_nt"] = out["guide"].map(looks_nt) | out["target"].map(looks_nt)
    return out

def get_program_matrix(adata, cnmf_key=CNMF_KEY, programs=PROGRAMS):
    usage = adata.obsm[cnmf_key]

    if isinstance(usage, pd.DataFrame):
        usage_df = usage.copy()
        usage_df.index = adata.obs_names
        out = {}
        for p in programs:
            possible = [p, str(p), f"P{p}", f"Program {p}", f"program {p}", f"Program_{p}"]
            found = None
            for c in usage_df.columns:
                if c in possible or str(c) in possible:
                    found = c
                    break
            if found is None:
                found = usage_df.columns[p - 1]
            out[f"Program {p}"] = usage_df[found].astype(float).values
        return pd.DataFrame(out, index=adata.obs_names)

    usage = np.asarray(usage)
    out = {}
    for p in programs:
        out[f"Program {p}"] = usage[:, p - 1].astype(float)
    return pd.DataFrame(out, index=adata.obs_names)

def zscore_by_day(program_df, obs, day_col="day_clean", background_mask=None, min_background=50):
    z = pd.DataFrame(index=program_df.index)
    day_series = obs[day_col].map(clean_day)

    if background_mask is None:
        background_mask = pd.Series(False, index=program_df.index)

    for day in sorted(day_series.unique()):
        day_mask = day_series.eq(day)
        bg = day_mask & background_mask
        if bg.sum() < min_background:
            bg = day_mask

        mu = program_df.loc[bg].mean(axis=0)
        sd = program_df.loc[bg].std(axis=0, ddof=1).replace(0, np.nan)
        z.loc[day_mask, program_df.columns] = (program_df.loc[day_mask] - mu) / sd

    return z.astype(float)

def log2_or(a, b, c, d, pc=0.5):
    return np.log2(((a + pc) / (b + pc)) / ((c + pc) / (d + pc)))

def cliffs_delta(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    # Efficient enough for sampled data; for very large arrays, downsample first.
    if len(x) * len(y) > 2_000_000:
        rng = np.random.default_rng(1)
        x = rng.choice(x, min(len(x), 1000), replace=False)
        y = rng.choice(y, min(len(y), 1000), replace=False)
    gt = sum((xi > y).sum() for xi in x)
    lt = sum((xi < y).sum() for xi in x)
    return (gt - lt) / (len(x) * len(y))



## 3. Build per-cell program scores

This cell assumes `adata` already exists.


In [ ]:

# Make sure adata exists before running this cell.
print(adata)

obs = adata.obs.copy()
obs.index = adata.obs_names
obs["day_clean"] = obs[DAY_COL].map(clean_day)
obs["cell_type_clean"] = obs[CELLTYPE_COL].astype(str)

guide_names = pd.Index([str(x) for x in adata.uns["guide_names"]])
guide_assignment = as_dense_bool(adata.obsm[GUIDE_ASSIGNMENT_KEY])

guide_target = build_guide_target_table(guide_names, GUIDE_TARGET_CSV)
guide_target.to_csv(os.path.join(OUTDIR, "guide_target_mapping_used.csv"), index=False)

nt_guides = guide_target.loc[guide_target["is_nt"], "guide"].tolist()
nt_idx = np.where(guide_names.isin(nt_guides))[0]
nt_carrier = guide_assignment[:, nt_idx].any(axis=1) if len(nt_idx) else np.zeros(adata.n_obs, dtype=bool)
nt_carrier = pd.Series(nt_carrier, index=adata.obs_names, name="non_targeting_carrier")

program_raw = get_program_matrix(adata)
program_z = zscore_by_day(program_raw, obs, background_mask=nt_carrier)

cell_df = pd.concat([obs[["day_clean", "cell_type_clean"]], program_z], axis=1)

# Fate-axis scores. Positive = alternative/failure bias.
cell_df["D1_alt_P13_minus_P30"] = cell_df["Program 13"] - cell_df["Program 30"]
cell_df["D2_alt_P17_minus_P15"] = cell_df["Program 17"] - cell_df["Program 15"]
cell_df["D2_YAPLat_minus_PreEC"] = (
    cell_df[["Program 17", "Program 34"]].mean(axis=1)
    - cell_df[["Program 15", "Program 23"]].mean(axis=1)
)
cell_df["D3_ALT_minus_EC"] = (
    cell_df[["Program 27", "Program 34", "Program 35"]].mean(axis=1)
    - cell_df[["Program 2", "Program 3", "Program 8", "Program 11"]].mean(axis=1)
)

cell_df["D1_program_state"] = np.where(cell_df["Program 13"] > cell_df["Program 30"], "P13_high", "P30_high")
cell_df["D2_program_state"] = np.where(cell_df["Program 17"] > cell_df["Program 15"], "P17_high", "P15_high")

# Add embedding if present
embed_key = None
for key in ["X_umap", "X_tsne", "umap", "tsne"]:
    if key in adata.obsm:
        embed_key = key
        break

if embed_key is not None:
    emb = np.asarray(adata.obsm[embed_key])
    cell_df["embed1"] = emb[:, 0]
    cell_df["embed2"] = emb[:, 1]
    cell_df["embedding_key"] = embed_key

cell_df.insert(0, "cell_id", cell_df.index)
cell_df.to_csv(os.path.join(OUTDIR, "all_cells_program_z_scores_and_states.csv"), index=False)

print(cell_df.shape)
cell_df.head()



## 4. Build target and guide carrier masks


In [ ]:

non_nt_targets = sorted(guide_target.loc[~guide_target["is_nt"], "target"].dropna().astype(str).unique())
target_to_guides = {
    t: guide_target.loc[guide_target["target"].astype(str).eq(t), "guide"].tolist()
    for t in non_nt_targets
}

target_carriers = {}
for target, guides in target_to_guides.items():
    idx = np.where(guide_names.isin(guides))[0]
    if len(idx):
        target_carriers[target] = pd.Series(guide_assignment[:, idx].any(axis=1), index=adata.obs_names)
    else:
        target_carriers[target] = pd.Series(False, index=adata.obs_names)

target_carrier_df = pd.DataFrame(target_carriers)
target_count = target_carrier_df.sum(axis=1)

hippo_present = [g for g in HIPPO_CORE if g in target_carriers]
print("Hippo targets present:", hippo_present)

carrier_rows = []
for t in non_nt_targets:
    carrier = target_carriers[t]
    row = {
        "target": t,
        "n_cells_all_days": int(carrier.sum()),
        "n_guides": len(target_to_guides[t]),
        "is_hippo_core": t in hippo_present,
    }
    for day in sorted(obs["day_clean"].unique()):
        row[f"n_cells_{day}"] = int((carrier & obs["day_clean"].eq(day)).sum())
    carrier_rows.append(row)

pd.DataFrame(carrier_rows).to_csv(os.path.join(OUTDIR, "target_carrier_counts.csv"), index=False)
pd.DataFrame(carrier_rows).head()



## 5. Export target-level fate-axis effects by day


In [ ]:

SCORES = [
    "D1_alt_P13_minus_P30",
    "D2_alt_P17_minus_P15",
    "D2_YAPLat_minus_PreEC",
    "D3_ALT_minus_EC",
]

def make_background(carrier, min_background=50):
    bg = nt_carrier & (~carrier)
    if bg.sum() < min_background:
        bg = ~carrier
    return pd.Series(bg.values if hasattr(bg, "values") else bg, index=adata.obs_names)

rows = []
for mode in ["all_carriers", "single_target_carriers"]:
    for target in non_nt_targets:
        carrier = target_carriers[target]
        if mode == "single_target_carriers":
            carrier = carrier & target_count.eq(1)

        bg_global = make_background(carrier)

        for day in sorted(obs["day_clean"].unique()):
            day_mask = obs["day_clean"].eq(day)
            cmask = carrier & day_mask
            bmask = bg_global & day_mask
            if bmask.sum() < 50:
                bmask = (~carrier) & day_mask

            row = {
                "mode": mode,
                "target": target,
                "day": day,
                "is_hippo_core": target in hippo_present,
                "n_carrier": int(cmask.sum()),
                "n_background": int(bmask.sum()),
            }

            for score in SCORES:
                x = cell_df.loc[cmask, score].dropna()
                y = cell_df.loc[bmask, score].dropna()
                row[f"{score}_carrier_mean"] = x.mean() if len(x) else np.nan
                row[f"{score}_background_mean"] = y.mean() if len(y) else np.nan
                row[f"{score}_effect_vs_background"] = row[f"{score}_carrier_mean"] - row[f"{score}_background_mean"]

                if len(x) >= 5 and len(y) >= 5:
                    row[f"{score}_ks_p"] = ks_2samp(x, y).pvalue
                    row[f"{score}_mw_p"] = mannwhitneyu(x, y, alternative="two-sided").pvalue
                    row[f"{score}_wasserstein"] = wasserstein_distance(x, y)
                    row[f"{score}_cliffs_delta"] = cliffs_delta(x.values, y.values)
                else:
                    row[f"{score}_ks_p"] = np.nan
                    row[f"{score}_mw_p"] = np.nan
                    row[f"{score}_wasserstein"] = np.nan
                    row[f"{score}_cliffs_delta"] = np.nan

            rows.append(row)

target_effects = pd.DataFrame(rows)
target_effects.to_csv(os.path.join(OUTDIR, "target_level_fate_axis_distribution_tests.csv"), index=False)
target_effects[target_effects["is_hippo_core"]].head()



## 6. Export cell-state enrichment summaries

These are the Nature-paper-style composition tests:
- D1 program-state P13-high vs P30-high
- D2 Lat-Meso vs Pre-EC
- D2 P17-high vs P15-high
- D3 alternative vs EC
- D3 mesenchymal vs EC


In [ ]:

def enrichment_rows_for_target(target, carrier, bg, mode):
    day = cell_df["day_clean"]
    ct = cell_df["cell_type_clean"]

    contrasts = [
        ("D1_program_P13high_vs_P30high", "d1",
         cell_df["D1_program_state"].eq("P13_high"),
         cell_df["D1_program_state"].eq("P30_high")),
        ("D2_celltype_LatMeso_vs_PreEC", "d2",
         ct.eq(D2_ALT), ct.eq(D2_PRE_EC)),
        ("D2_program_P17high_vs_P15high", "d2",
         cell_df["D2_program_state"].eq("P17_high"),
         cell_df["D2_program_state"].eq("P15_high")),
        ("D3_celltype_AltBroad_vs_EC", "d3",
         ct.isin(D3_ALT_LABELS), ct.isin(D3_EC_LABELS)),
        ("D3_celltype_FibroMesen_vs_EC", "d3",
         ct.isin(D3_MES_LABELS), ct.isin(D3_EC_LABELS)),
    ]

    out = []
    for contrast, dday, event_mask, ref_mask in contrasts:
        valid_day = day.eq(dday)
        event = event_mask & valid_day
        ref = ref_mask & valid_day

        a = int((carrier & event).sum())
        b = int((carrier & ref).sum())
        c = int((bg & event).sum())
        d = int((bg & ref).sum())

        n_target = a + b
        n_bg = c + d

        if n_target and n_bg:
            lor = log2_or(a, b, c, d)
            pval = fisher_exact([[a, b], [c, d]])[1]
        else:
            lor = np.nan
            pval = np.nan

        out.append({
            "mode": mode,
            "target": target,
            "contrast": contrast,
            "day": dday,
            "is_hippo_core": target in hippo_present,
            "target_event_n": a,
            "target_ref_n": b,
            "background_event_n": c,
            "background_ref_n": d,
            "target_total_event_ref": n_target,
            "background_total_event_ref": n_bg,
            "target_event_fraction": a / n_target if n_target else np.nan,
            "background_event_fraction": c / n_bg if n_bg else np.nan,
            "log2OR_event_vs_ref": lor,
            "fisher_exact_p": pval,
        })
    return out

enrich_rows = []
for mode in ["all_carriers", "single_target_carriers"]:
    for target in non_nt_targets:
        carrier = target_carriers[target]
        if mode == "single_target_carriers":
            carrier = carrier & target_count.eq(1)
        bg = make_background(carrier)
        enrich_rows.extend(enrichment_rows_for_target(target, carrier, bg, mode))

enrich_df = pd.DataFrame(enrich_rows)
enrich_df.to_csv(os.path.join(OUTDIR, "state_enrichment_target_level.csv"), index=False)
enrich_df[enrich_df["is_hippo_core"]].head(20)



## 7. Export guide-level pseudo-replicate effects

This is important because guides can be treated as pseudo-replicates, closer to the individual-organoid logic in the Nature paper.


In [ ]:

guide_rows = []
for gi, guide in enumerate(guide_names):
    gt_row = guide_target.loc[guide_target["guide"].eq(guide)].iloc[0]
    target = str(gt_row["target"])
    is_nt = bool(gt_row["is_nt"])

    if is_nt:
        continue

    carrier = pd.Series(guide_assignment[:, gi], index=adata.obs_names)
    bg = make_background(carrier)

    for day in sorted(obs["day_clean"].unique()):
        day_mask = obs["day_clean"].eq(day)
        cmask = carrier & day_mask
        bmask = bg & day_mask
        if bmask.sum() < 50:
            bmask = (~carrier) & day_mask

        row = {
            "guide": guide,
            "target": target,
            "day": day,
            "is_hippo_core": target in hippo_present,
            "n_carrier": int(cmask.sum()),
            "n_background": int(bmask.sum()),
        }

        for score in SCORES:
            x = cell_df.loc[cmask, score].dropna()
            y = cell_df.loc[bmask, score].dropna()
            row[f"{score}_carrier_mean"] = x.mean() if len(x) else np.nan
            row[f"{score}_background_mean"] = y.mean() if len(y) else np.nan
            row[f"{score}_effect_vs_background"] = row[f"{score}_carrier_mean"] - row[f"{score}_background_mean"]

        guide_rows.append(row)

guide_effects = pd.DataFrame(guide_rows)
guide_effects.to_csv(os.path.join(OUTDIR, "guide_level_axis_effects.csv"), index=False)
guide_effects[guide_effects["is_hippo_core"]].head()



## 8. Export Hippo carrier cell distributions and UMAP overlays


In [ ]:

rng = np.random.default_rng(1)

hippo_cell_rows = []
umap_rows = []

for target in hippo_present:
    carrier = target_carriers[target]
    bg = make_background(carrier)

    # Include all Hippo carrier cells and sampled background
    carrier_cells = cell_df.index[carrier].to_numpy()
    bg_cells = cell_df.index[bg].to_numpy()
    if len(bg_cells) > 15000:
        bg_cells = rng.choice(bg_cells, size=15000, replace=False)

    selected = np.concatenate([carrier_cells, bg_cells])

    cols = [
        "cell_id", "day_clean", "cell_type_clean",
        "D1_alt_P13_minus_P30", "D2_alt_P17_minus_P15",
        "D2_YAPLat_minus_PreEC", "D3_ALT_minus_EC",
        "D1_program_state", "D2_program_state",
        "Program 13", "Program 30", "Program 15", "Program 17",
        "Program 23", "Program 34", "Program 2", "Program 3",
        "Program 8", "Program 11", "Program 27", "Program 35",
    ]

    if "embed1" in cell_df.columns:
        cols += ["embed1", "embed2", "embedding_key"]

    tmp = cell_df.loc[selected, cols].copy()
    tmp["target"] = target
    tmp["is_carrier"] = tmp.index.isin(carrier_cells)
    hippo_cell_rows.append(tmp)

    if "embed1" in cell_df.columns:
        # Smaller UMAP overlay file by day
        for day in ["d1", "d2", "d3"]:
            day_mask = cell_df["day_clean"].eq(day)
            carrier_day = cell_df.index[carrier & day_mask].to_numpy()
            bg_day = cell_df.index[(~carrier) & day_mask].to_numpy()
            if len(bg_day) > 5000:
                bg_day = rng.choice(bg_day, size=5000, replace=False)
            sel = np.concatenate([carrier_day, bg_day])
            u = cell_df.loc[sel, ["cell_id", "day_clean", "cell_type_clean", "embed1", "embed2"]].copy()
            u["target"] = target
            u["is_carrier"] = u.index.isin(carrier_day)
            umap_rows.append(u)

if hippo_cell_rows:
    hippo_cells = pd.concat(hippo_cell_rows, axis=0)
    hippo_cells.to_csv(os.path.join(OUTDIR, "hippo_cell_score_distributions_for_plotting.csv"), index=False)
    print("Wrote hippo_cell_score_distributions_for_plotting.csv", hippo_cells.shape)

if umap_rows:
    hippo_umap = pd.concat(umap_rows, axis=0)
    hippo_umap.to_csv(os.path.join(OUTDIR, "hippo_umap_overlay_for_plotting.csv"), index=False)
    print("Wrote hippo_umap_overlay_for_plotting.csv", hippo_umap.shape)



## 9. Guide convergence summary

This summarizes whether Hippo guides tend to move in the same direction on the fate axes.


In [ ]:

conv_rows = []
hippo_guides = guide_effects[guide_effects["is_hippo_core"]].copy()

for day in sorted(hippo_guides["day"].unique()):
    sub_day = hippo_guides[hippo_guides["day"].eq(day)]
    for score in SCORES:
        eff_col = f"{score}_effect_vs_background"
        vals = sub_day[eff_col].dropna()

        if len(vals) == 0:
            continue

        conv_rows.append({
            "day": day,
            "score": score,
            "n_guides": len(vals),
            "mean_effect": vals.mean(),
            "median_effect": vals.median(),
            "fraction_positive": (vals > 0).mean(),
            "fraction_negative": (vals < 0).mean(),
            "one_sample_t_p": stats.ttest_1samp(vals, 0).pvalue if len(vals) >= 2 else np.nan,
            "signed_rank_p": stats.wilcoxon(vals).pvalue if len(vals) >= 2 and not np.all(vals == 0) else np.nan,
        })

conv_df = pd.DataFrame(conv_rows)
conv_df.to_csv(os.path.join(OUTDIR, "hippo_guide_convergence_summary.csv"), index=False)
conv_df



## 10. Manifest


In [ ]:

manifest = pd.DataFrame({
    "file": sorted(os.listdir(OUTDIR)),
    "path": [os.path.abspath(os.path.join(OUTDIR, fn)) for fn in sorted(os.listdir(OUTDIR))]
})
manifest.to_csv(os.path.join(OUTDIR, "export_manifest.csv"), index=False)
manifest
